# **Image Classification Cat or Dog Model Deployment**

# **Nama : Doni Firmansyah**

In [1]:
# @markdown Import Library

# !pip install split-folders

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.preprocessing import image
import os
from google.colab import files
import pathlib

# import splitfolders

In [ ]:
# @markdown Collecting Data :\
# @markdown Dataset yang digunakan adalah dari https://www.kaggle.com/datasets/dhondoi/big-dataset-dog-vs-cat

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d dhondoi/big-dataset-dog-vs-cat --unzip
# !unzip big-dataset-dog-vs-cat.zip

 30% 780M/2.58G [00:30<01:03, 30.5MB/s]

In [ ]:
# @markdown Collecting Data :\
# @markdown Checking Total Dataset

ANIMAL = []

print("---------------")
print("Before Splitting")

DIR = "dog_vs_cat"
TOTAL_OUTPUT = len(os.listdir(DIR))
totalAll = 0

for labelName in os.listdir(DIR):
  ANIMAL.append(labelName)

  dirLabel = os.path.join(DIR, labelName)
  total = len([name for name in os.listdir(dirLabel) if os.path.isfile(os.path.join(dirLabel, name))])

  print(f"Count Image {labelName} : {total}")

  totalAll+=total

print(f"\nTotal Image : {totalAll}")
print(f"Label Names : {ANIMAL}")
print("---------------")

In [ ]:
# @markdown Cleaning Data :\
# @markdown Check abnormal image

from imageio.v2 import imread
import warnings
warnings.filterwarnings("error")

DIR = "pet_animal"
TOTAL_OUTPUT = len(os.listdir(DIR))

print("Before Splitting")
for labelName in os.listdir(DIR):
  dirLabel = os.path.join(DIR, labelName)
  total = len([name for name in os.listdir(dirLabel) if os.path.isfile(os.path.join(dirLabel, name))])
  print(f"Total Dataset {labelName} : {total}")
  try:
    for name in os.listdir(dirLabel):
      if len(imread(os.path.join(dirLabel, name)).shape) != 3:
        raise ValueError(f"{dirLabel} : {name} Not RGB!")
  except RuntimeWarning:
    breakpoint()
  except ValueError:
    breakpoint()
print("---------------")

In [ ]:
# @markdown Collecting Data :\
# @markdown Split to The Data Train and Validation

# DIR_ZIPPED = "dog_vs_cat"
# DIR = "dog_vs_cat/dev"

# splitfolders.ratio(
#     DIR_ZIPPED,
#     DIR,
#     seed=1, ratio=(0.8,0.2)
#     )

# TRAIN_DIR = os.path.join(DIR, "train")
# VALIDATION_DIR = os.path.join(DIR, "val")

# print("After Splitting")
# print(f"\nDirectory For Training and Validation  : {os.listdir(DIR)}")

# print(f"\nInside Directory train  : {os.listdir(TRAIN_DIR)}")
# dirDog = os.path.join(TRAIN_DIR, "dogs")
# totalDog = len([name for name in os.listdir(dirDog) if os.path.isfile(os.path.join(dirDog, name))])
# print(f"Total Dataset Train Dog : {totalDog}")
# dirCat = os.path.join(TRAIN_DIR, "cats")
# totalCat = len([name for name in os.listdir(dirCat) if os.path.isfile(os.path.join(dirCat, name))])
# print(f"Total Dataset Train Cat : {totalCat}")

# print(f"\nInside Directory val  : {os.listdir(VALIDATION_DIR)}")
# dirDog = os.path.join(VALIDATION_DIR, "dogs")
# totalDog = len([name for name in os.listdir(dirDog) if os.path.isfile(os.path.join(dirDog, name))])
# print(f"Total Dataset Train Dog : {totalDog}")
# dirCat = os.path.join(VALIDATION_DIR, "cats")
# totalCat = len([name for name in os.listdir(dirCat) if os.path.isfile(os.path.join(dirCat, name))])
# print(f"Total Dataset Train Cat : {totalCat}")

# print("---------------")

In [ ]:
# os. rmdir(f'{TRAIN_DIR}/.ipynb_checkpoints')
# os. rmdir(f'{VALIDATION_DIR}/.ipynb_checkpoints')

In [ ]:
# @markdown Pre-Processing Data :\
# @markdown Image Augmentation

trainIDG = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.1,
    shear_range=0.2,
    fill_mode="nearest",
    validation_split=0.2
)
validationIDG = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

In [ ]:
# @markdown Pre-Processing Data :\
# @markdown Creating Callback

class CustomCallback(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):
        if logs.get("accuracy") > 0.92 and logs.get("val_accuracy") > 0.92:
            print(f"Training and Validation on epoch {epoch+1} get minimum accuracy (92%) : Train {logs.get('accuracy'):.2f} Validation {logs.get('val_accuracy'):.2f}")
            self.model.stop_training = True

In [ ]:
# @markdown Pre-Processing Data :\
# @markdown Modelling

WIDTH = 64
HEIGHT = 64

model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Conv2D(
            16, (3, 3), activation="relu", input_shape=(WIDTH, HEIGHT, 3)
        ),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

model.summary()

In [ ]:
# @markdown Pre-Processing Data:\
# @markdown Generate Train and Validation Dataset

BATCH_SIZE = 32

print("After Splitting Using Image Data Generator")
print("\nDataset Train")
trainDataset = trainIDG.flow_from_directory(DIR,
                                            class_mode='binary',
                                            target_size=(WIDTH, HEIGHT),
                                            subset="training",
                                            batch_size=BATCH_SIZE,
                                            seed=42
                                            )

print("\nDataset Validation")
validationDataset = validationIDG.flow_from_directory(DIR,
                                                      class_mode='binary',
                                                      target_size=(WIDTH, HEIGHT),
                                                      subset="validation",
                                                      batch_size=BATCH_SIZE,
                                                      seed=42
                                                      )
print("---------------")

In [ ]:
# @markdown **Pre-Processing Data** :\
# @markdown Compiler for Model

LEARNING_RATE = 0.001

model.compile(loss='binary_crossentropy',
              optimizer=tf.optimizers.Adam(learning_rate=LEARNING_RATE),
              metrics=['accuracy'])

In [ ]:
# @markdown Processing Data :\
# @markdown Fitting

EPOCH = 100

history = model.fit(trainDataset,epochs=EPOCH,verbose=2,validation_data=validationDataset,callbacks=[CustomCallback()])

In [ ]:
# @markdown Post-Processing :\
# @markdown Accuracy Visualization Result

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('ACCURACY')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# @markdown Post-Processing :\
# @markdown Loss Visualization Result

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('LOSS')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# @markdown Post-Processing :\
# @markdown Predict

uploaded = files.upload()

for fn in uploaded.keys():
  path = fn
  img = image.load_img(path, target_size =(150,150))
  plt.imshow(img)
  plt.show()
  x = image.img_to_array(img)
  x = np.expand_dims(x, axis=0)

  images = np.vstack([x])
  classes = model.predict(images, batch_size=BATCH_SIZE)


  print(fn)
  if classes[0,0]!=0:
    print('Dog')
  else:
    print('Cat')

In [ ]:
# @markdown Post-Processing :\
# @markdown Saving Model

export_dir = 'saved_model/'
tf.saved_model.save(model, export_dir)


converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()


tflite_model_file = pathlib.Path('catvsdog.tflite')
tflite_model_file.write_bytes(tflite_model)